# WAVE analytical-verification figure

This notebook creates the main-paper WAVE figures. It visibly regenerates every required WAVE benchmark with the current source on a clean checkout; set `ENSURE_CURRENT_RESULTS=False` only when restyling existing audited results.


## Reproducible environment

The first code cell installs the repository's validation package and Python dependencies into the active kernel. A clean checkout also needs GNU Make and gfortran to compile the selected solver on first use.


In [ ]:
from pathlib import Path
SEARCH_ROOT = Path.cwd().resolve()
REPOSITORY = next(candidate for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents) if (candidate / 'validation' / 'pyproject.toml').is_file())
%pip install -q -e {REPOSITORY / 'validation'}


In [ ]:
import os
from avac4qgis_validation import validation_case
case = validation_case('WAVE', 'Paper_figures')
CORES = max(1, os.cpu_count() or 1)
case.path


In [ ]:
ENSURE_CURRENT_RESULTS = True
WAVE_ROOT = case.path.parent
if ENSURE_CURRENT_RESULTS:
    for CASE_KEY in ('transcritical_shock', 'macdonald_smooth_shock', 'ritter_dry_dam_break', 'thacker_planar_paraboloid'):
        case.run(WAVE_ROOT / 'run_validation.py', CASE_KEY, '--solver', 'wave', '--output-root', WAVE_ROOT, '--cores', CORES, cwd=WAVE_ROOT)
    case.run(WAVE_ROOT / '07_baines_flow_over_bump' / 'run_baines_validation.py', cwd=WAVE_ROOT / '07_baines_flow_over_bump')
    case.run(WAVE_ROOT / '08_amr_parallel' / 'run_amr_parallel_validation.py', cwd=WAVE_ROOT / '08_amr_parallel')
    wrr_driver = WAVE_ROOT.parents[1] / 'AVAC' / '2008_WRR_sloping_bed' / 'run_avac_validation.py'
    case.run(wrr_driver, '--solver', 'wave', '--output-root', WAVE_ROOT / '2008_WRR_sloping_bed', '--dx', 0.005, '--t-final', 5.0, '--nout', 100, '--cores', CORES, '--xlower', -10.0, '--xupper', 40.0, '--rear-tracker', 'tutorial', cwd=WAVE_ROOT / '2008_WRR_sloping_bed')
case.run('make_wave_verification_figures.py')
case.run('make_wave_appendix_figures.py')


In [ ]:
case.show('../../../docs/article/figures/wave_analytical_verification.png', '../../../docs/article/figures/wave_additional_benchmarks.png', '../../../docs/article/figures/wave_numerical_diagnostics.png')
